# W2-D2 RCA assignment

This notebook builds a small RCA pipeline using the D1 cluster summary, service graph, and incident history. It uses graph traversal + temporal scoring for service candidates, and TF-IDF retrieval over incident summaries as the bonus path. The final output is written to `results/rca_output.json`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path('dataset')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def build_service_graph(definition: dict) -> nx.DiGraph:
    graph = nx.DiGraph()
    for node in definition.get('services', []) + definition.get('stores', []):
        graph.add_node(node['name'], **node)
    for edge in definition.get('edges', []):
        graph.add_edge(edge['from'], edge['to'], **{k: v for k, v in edge.items() if k not in ['from', 'to']})
    return graph

def build_cluster_query(cluster: dict) -> str:
    services_text = ' '.join(cluster.get('services', []))
    fingerprint_text = ' '.join(cluster.get('fingerprints', []))
    severity_text = cluster.get('max_severity', '')
    return ' '.join([services_text, fingerprint_text, severity_text]).strip()

def incident_text(incident: dict) -> str:
    parts = [incident.get('root_cause_class', '')] + incident.get('affected_services', [])
    parts += incident.get('log_signatures', [])
    metric_text = [' '.join([m.get('service', ''), m.get('metric', ''), m.get('delta', '')]) for m in incident.get('metric_signatures', [])]
    trace_text = [' '.join([t.get('from', ''), t.get('to', ''), str(t.get('p99_deviation_ratio', ''))]) for t in incident.get('trace_signatures', [])]
    return ' '.join(parts + metric_text + trace_text).strip()

def normalize_scores(scores: dict) -> dict:
    if not scores:
        return {}
    max_value = max(scores.values())
    min_value = min(scores.values())
    if max_value == min_value:
        return {k: 1.0 for k in scores}
    return {k: (v - min_value) / (max_value - min_value) for k, v in scores.items()}

def score_graph_candidates(graph: nx.DiGraph, cluster: dict) -> list[tuple[str, float]]:
    cluster_services = cluster.get('services', [])
    graph_scores = {}
    for service in cluster_services:
        graph_scores[service] = graph.in_degree(service) + graph.out_degree(service)
    graph_norm = normalize_scores(graph_scores)

    fp_counts = Counter(fp.split('|')[0] for fp in cluster.get('fingerprints', []))
    severity_weights = {'warn': 1.0, 'crit': 2.0}
    temporal_scores = {}
    for service in cluster_services:
        temporal_scores[service] = 0.0
    for fp in cluster.get('fingerprints', []):
        service_name, metric_name, severity = fp.split('|')
        temporal_scores[service_name] += 1.0 * severity_weights.get(severity, 1.0)
    temporal_norm = normalize_scores(temporal_scores)

    combined = {}
    for service in cluster_services:
        combined[service] = 0.55 * graph_norm.get(service, 0.0) + 0.45 * temporal_norm.get(service, 0.0)

    return sorted(combined.items(), key=lambda item: item[1], reverse=True)[:3]

def build_tfidf_retriever(incidents: list[dict]) -> tuple[TfidfVectorizer, list[str], list[dict]]:
    texts = [incident_text(i) for i in incidents]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix, incidents

def retrieve_similar_incidents(query: str, vectorizer: TfidfVectorizer, matrix, incidents: list[dict], top_k: int = 3) -> list[tuple[dict, float]]:
    if not query.strip():
        return []
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, matrix)[0] if matrix.shape[0] > 0 else []
    ranked = sorted(list(enumerate(sims)), key=lambda item: item[1], reverse=True)[:top_k]
    return [(incidents[idx], float(score)) for idx, score in ranked if score > 0.0]

service_graph = build_service_graph(load_json(DATA_DIR / 'services.json'))
cluster_summary = load_json(Path('..') / 'd1' / 'results' / 'cluster_summary.json') if False else load_json(Path('..') / 'd1' / 'results' / 'cluster_summary.json')
incident_history = load_json(DATA_DIR / 'incidents_history.json')
vectorizer, incident_matrix, incidents = build_tfidf_retriever(incident_history)

print(f'Loaded graph with {service_graph.number_of_nodes()} nodes and {service_graph.number_of_edges()} edges')
print(f"Loaded {len(cluster_summary.get('clusters', []))} clusters from D1 summary")

In [ ]:
def explain_similarity(cluster: dict, candidates: list[tuple[dict, float]]) -> str:
    if not candidates:
        return 'No strong history match found; fallback to graph evidence.'
    top_ids = [item[0]['id'] for item in candidates]
    return f"Similar incidents are {', '.join(top_ids)} with TF-IDF evidence from affected services and signature text."

results = []
for cluster in cluster_summary.get('clusters', []):
    graph_top3 = score_graph_candidates(service_graph, cluster)
    top_service = graph_top3[0][0] if graph_top3 else None
    query = build_cluster_query(cluster)
    similar = retrieve_similar_incidents(query, vectorizer, incident_matrix, incidents, top_k=3)
    if similar:
        top_incident, top_sim = similar[0]
        root_cause_class = top_incident['root_cause_class']
        actions = top_incident.get('actions_taken', []) or ['page_oncall:platform-team']
        top_incident_id = top_incident['id']
    else:
        root_cause_class = 'unknown'
        actions = ['page_oncall:platform-team']
        top_sim = 0.0
        top_incident_id = 'none'

    max_graph_score = graph_top3[0][1] if graph_top3 else 0.0
    severity_score = 1.0 if cluster.get('max_severity') == 'crit' else 0.5
    confidence = min(1.0, 0.35 * max_graph_score + 0.55 * top_sim + 0.1 * severity_score)

    reasoning = (
        f'Graph traversal rated {top_service} highest in the impact path. '
        f'TF-IDF retrieval selected {top_incident_id} as the best historical match. '
        f'Combined evidence supports {top_service} as root cause with class {root_cause_class}.'
    )

    results.append({
        'cluster_id': cluster.get('cluster_id'),
        'graph_top3': [[svc, round(score, 3)] for svc, score in graph_top3],
        'root_cause': top_service or 'unknown',
        'class': root_cause_class,
        'confidence': round(confidence, 2),
        'actions': actions,
        'reasoning': reasoning,
        'similar_incidents': [item[0]['id'] for item in similar],
        'method': 'graph+tfidf',
    })

print('Analysis complete; preview of results:')
for row in results:
    print(row['cluster_id'], row['root_cause'], row['class'], row['confidence'], row['similar_incidents'])

In [ ]:
rca_output = {'clusters_analyzed': len(results), 'results': results}
output_path = RESULTS_DIR / 'rca_output.json'
with output_path.open('w', encoding='utf-8') as f:
    json.dump(rca_output, f, indent=2, ensure_ascii=False)

print(f'Wrote RCA output to {output_path}')
print(json.dumps(rca_output, indent=2, ensure_ascii=False)[:800])